# Sketch2Shape - isolated sketch-to-image experiment
Goal: one realistic chair PNG that preserves the uploaded drawing's structure, using pretrained **ControlNet Scribble 1.1 + Stable Diffusion 1.5**. This notebook does **not** invoke TripoSR or generate a GLB. Use a separate disposable **T4 GPU** Colab session so optional diffusion dependencies do not disturb the working reconstruction runtime.

This recipe targets Python 3.12, **torch 2.11.0+cu128**, Transformers **4.40.2**, Tokenizers **0.19.1**, Hub **0.25.2**, NumPy **1.26.4**, Diffusers **0.30.3**, Accelerate **0.34.2**. The package metadata constraints are compatible, but this combination with the newer torch has **not been GPU-tested here**. Incompatible preinstalled protected packages stop setup rather than trigger broad upgrades. No model training, paid API, hosted UI or tunnel.

In [ ]:
import os, sys, json, subprocess, tempfile, uuid, shutil, re
from pathlib import Path
import importlib.metadata as metadata

def run_checked(command, **kwargs):
    try:
        return subprocess.run(command, check=True, **kwargs)
    except (subprocess.CalledProcessError, subprocess.TimeoutExpired, OSError) as exc:
        raise RuntimeError("Experiment command failed. Inspect the output above; no automatic broad package upgrade or fallback was attempted.") from exc

if sys.version_info[:2] != (3, 12):
    raise RuntimeError("This recipe targets Python 3.12; select a compatible Colab runtime.")
gpu_probe = """
import json, torch
assert torch.__version__ == '2.11.0+cu128', f'Expected working torch 2.11.0+cu128, found {torch.__version__}; do not replace it blindly'
assert torch.cuda.is_available(), 'Select a GPU runtime'
assert torch.version.cuda == '12.8'
name = torch.cuda.get_device_name(0)
assert 'T4' in name, f'This first experiment targets T4, found {name}'
print(json.dumps({'torch': torch.__version__, 'torch_cuda': torch.version.cuda, 'gpu': name}))
"""
run_checked([sys.executable, "-c", gpu_probe], timeout=60)
print("Python:", sys.version)
run_checked(["nvidia-smi"], timeout=30)
if shutil.which("nvcc"):
    print(subprocess.check_output(["nvcc", "--version"], text=True))
# No kernel torch/cv2/NumPy imports: later checks/inference use fresh processes.

## Load current source using ZIP or Drive
Create a NEW ZIP containing updated `backend/`, `examples/`, and `notebooks/Sketch2Shape_Sketch_Experiment.ipynb`, preserving those folder names. Exclude virtual environments, caches, `.git`, weights, outputs and credentials. The old backend ZIP does not contain this stage. Windows path separators are normalized safely; existing source directories are never overwritten.

In [ ]:
import shutil
import stat
import zipfile
from pathlib import Path, PurePosixPath

def extract_project_zip(archive_file, destination):
    """Normalize Windows ZIP separators, reject unsafe paths, never overwrite."""
    root = Path(destination).resolve()
    if root.exists():
        raise FileExistsError(f"Extraction target already exists: {root}")
    with zipfile.ZipFile(archive_file) as archive:
        entries = archive.infolist()
        if sum(entry.file_size for entry in entries) > 512 * 1024**2:
            raise ValueError("ZIP exceeds 512 MiB unpacked; omit environments/weights/outputs")
        normalized = []
        seen = set()
        for entry in entries:
            name = entry.filename.replace("\\", "/")
            parts = PurePosixPath(name).parts
            if (not parts or name.startswith("/") or ".." in parts
                    or any(":" in part for part in parts)
                    or stat.S_ISLNK(entry.external_attr >> 16)):
                raise ValueError(f"Unsafe ZIP member: {entry.filename!r}")
            target = root.joinpath(*parts)
            if not target.resolve().is_relative_to(root):
                raise ValueError("ZIP path escapes extraction directory")
            if target in seen:
                raise ValueError("ZIP contains duplicate normalized paths")
            seen.add(target)
            normalized.append((entry, target, name.endswith("/")))
        root.mkdir(parents=True)
        for entry, target, is_directory in normalized:
            if is_directory:
                target.mkdir(parents=True, exist_ok=True)
            else:
                target.parent.mkdir(parents=True, exist_ok=True)
                with archive.open(entry) as source, target.open("xb") as output:
                    shutil.copyfileobj(source, output)
    matches = list(root.rglob("backend/generator.py"))
    if len(matches) != 1:
        raise ValueError(f"Expected one project backend/generator.py, found {len(matches)}")
    return matches[0].parent.parent.resolve()

In [ ]:
SOURCE_MODE = "zip"  # or "drive"
DRIVE_PROJECT = "/content/drive/MyDrive/Sketch2Shape"
if SOURCE_MODE == "drive":
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT = Path(DRIVE_PROJECT).resolve()
elif SOURCE_MODE == "zip":
    from google.colab import files
    import io
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError("Upload exactly one project ZIP")
    target = Path(tempfile.mkdtemp(prefix="sketch-experiment-", dir="/content")) / "source"
    PROJECT = extract_project_zip(io.BytesIO(next(iter(uploaded.values()))), target)
    del uploaded
else:
    raise ValueError("SOURCE_MODE must be zip or drive")
for required in ("backend/sketch_preprocess.py", "backend/sketch_refiner.py", "backend/requirements-sketch-colab.txt"):
    if not (PROJECT / required).is_file():
        raise FileNotFoundError(f"Upload the NEW source; missing {required}")
os.chdir(PROJECT)
print("Project:", PROJECT)

## Dependencies without replacing the working stack
Protect torch, Transformers, Tokenizers and Hugging Face Hub. Existing conflicting versions stop with a clear error; missing Hugging Face packages can be installed at the declared versions. Diffusers 0.30.3 requires Hub >=0.23.2, which includes 0.25.2. No `pip install -U`, xFormers, controlnet_aux, rembg or torchmcubes is required for this stage. This is a compatibility recipe, not a tested full-environment lock.

If the preflight fails, use a separate environment configured with these versions; do not repair the working TripoSR session by broad upgrades. Conflicting optional preinstalled packages (for example newer PEFT) may still break imports; the import smoke check must pass before any weight download.

In [ ]:
PROTECTED = {"torch": "2.11.0+cu128", "transformers": "4.40.2", "tokenizers": "0.19.1", "huggingface-hub": "0.25.2"}
conflicts = []
for package, expected in PROTECTED.items():
    try:
        actual = metadata.version(package)
    except metadata.PackageNotFoundError:
        if package == "torch":
            conflicts.append("torch is missing")
        continue
    if actual != expected:
        conflicts.append(f"{package}: expected {expected}, found {actual}")
if conflicts:
    raise RuntimeError("Protected dependency mismatch. No package changes made: " + "; ".join(conflicts))
constraints = Path(tempfile.gettempdir()) / f"sketch-protected-{uuid.uuid4().hex}.txt"
constraints.write_text("\n".join(f"{name}=={version}" for name, version in PROTECTED.items()) + "\n")
run_checked([sys.executable, "-m", "pip", "install", "-c", str(constraints),
             "-r", str(PROJECT / "backend/requirements-sketch-colab.txt")], timeout=900)
check_code = """
from backend.sketch_refiner import check_refinement_environment
print(check_refinement_environment())
from diffusers import ControlNetModel, StableDiffusionControlNetPipeline, UniPCMultistepScheduler
import torch, cv2, numpy
assert torch.cuda.is_available()
assert hasattr(StableDiffusionControlNetPipeline, 'enable_model_cpu_offload')
print('Diffusers/ControlNet imports passed; no models loaded')
"""
run_checked([sys.executable, "-c", check_code], cwd=str(PROJECT), timeout=120)

## Pretrained model access and licenses
The requested [Scribble checkpoint](https://huggingface.co/lllyasviel/control_v11p_sd15_scribble) and [SD 1.5 mirror](https://huggingface.co/stable-diffusion-v1-5/stable-diffusion-v1-5) are publicly downloadable at implementation time. SD 1.5 uses **CreativeML OpenRAIL-M**; the ControlNet card labels OpenRAIL and links CreativeML OpenRAIL-M. Review their terms and use restrictions. Diffusers and Accelerate use Apache-2.0. No weights are trained or modified.

Normally no token is needed. If access changes or a 401/403 occurs, review the model page and authenticate using your own Hugging Face account/Colab secret; never paste tokens into source or outputs. The optional cell below only reads a user-managed secret when explicitly enabled. Model access is not bypassed, and no paid inference provider is used.

In [ ]:
USE_HF_SECRET = False
if USE_HF_SECRET:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    print("Hugging Face token supplied from Colab Secrets (not displayed).")

In [ ]:
run_checked([sys.executable, "-m", "compileall", "-q", "backend"], cwd=str(PROJECT), timeout=60)
run_checked([sys.executable, "-m", "unittest", "discover", "-s", "backend/tests", "-v"], cwd=str(PROJECT), timeout=180)

## Upload one hand-drawn chair sketch
Use PNG/JPEG. Transparent, RGB/RGBA and grayscale canvases are supported. The entire image is fitted into 512x512 with padding, not stretched/cropped. Cleaning uses mild contrast/noise handling and removes only isolated single-pixel specks. No stroke completion or geometry invention is performed.

**Conditioning polarity: white strokes on black.** This follows the checkpoint's [control image example](https://huggingface.co/lllyasviel/control_v11p_sd15_scribble/resolve/main/images/control.png) and the author's [scribble demo](https://github.com/lllyasviel/ControlNet/blob/main/gradio_scribble2image.py), which sets dark input strokes to 255 in a zero-initialized map. The human-readable cleaned sketch remains dark-on-white. Inspect all three images before generation; stop if meaningful lines disappeared.

In [ ]:
from google.colab import files
uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError("Upload exactly one chair sketch PNG/JPEG")
name, data = next(iter(uploaded.items()))
input_root = PROJECT / "examples"
input_root.mkdir(exist_ok=True)
SKETCH = input_root / f"chair_sketch_{uuid.uuid4().hex}{Path(name).suffix.lower()}"
SKETCH.write_bytes(data)
del uploaded, data
OUTPUT_DIR = PROJECT / "outputs" / "sketch_refinements"
preview_record = Path(tempfile.gettempdir()) / f"sketch-preview-{uuid.uuid4().hex}.json"
preview_code = """
import json, sys
from pathlib import Path
from dataclasses import asdict
from backend.sketch_preprocess import preprocess_sketch
artifacts = preprocess_sketch(sys.argv[1], sys.argv[2])
Path(sys.argv[3]).write_text(json.dumps(asdict(artifacts)))
"""
run_checked([sys.executable, "-c", preview_code, str(SKETCH), str(OUTPUT_DIR), str(preview_record)], cwd=str(PROJECT), timeout=60)
preview = json.loads(preview_record.read_text())
from IPython.display import display, Image as DisplayImage
for label, key in (("Original sketch", "original_path"), ("Cleaned dark-on-white sketch", "cleaned_path"),
                   ("ControlNet conditioning: white on black", "conditioning_path")):
    print(label)
    display(DisplayImage(filename=preview[key]))
print(Path(preview["metadata_path"]).read_text())

## Describe the same chair, material and view
Example: `A studio product photograph of a single oak wooden chair, matching the drawn backrest, seat and leg arrangement, three-quarter view, isolated on a neutral light gray background, realistic wood texture.` Match your actual drawing's viewpoint/material. Avoid requesting a different silhouette. Inspect fidelity rather than assuming a photorealistic result is faithful.

In [ ]:
PROMPT = input("Describe your chair, material and viewpoint: ").strip()
if len(PROMPT) < 3:
    raise ValueError("A meaningful description is required")
SEED = 42
STEPS = 20
CFG = 7.0
CONTROL_STRENGTH = 1.0
NEGATIVE_PROMPT = "multiple chairs, duplicate objects, extra legs, text, watermark, blurry, distorted geometry"
os.environ.update({"SKETCH_SEED": str(SEED), "SKETCH_STEPS": str(STEPS), "SKETCH_CFG": str(CFG),
                   "SKETCH_CONTROL_STRENGTH": str(CONTROL_STRENGTH), "SKETCH_NEGATIVE_PROMPT": NEGATIVE_PROMPT})
print({"seed": SEED, "steps": STEPS, "cfg": CFG, "control_strength": CONTROL_STRENGTH, "prompt": PROMPT})

## Explicit GPU inference (first call downloads pretrained weights)
The backend loads official Diffusers models in FP16, uses Accelerate model CPU offload, UniPC sampling, and keeps the safety checker. A seeded CPU RNG is used; exact reproducibility across GPU/software versions is not guaranteed. No model is loaded at import. This cell calls only `refine_sketch()`, never `generate_3d()`.

In [ ]:
REFINED = None  # Failed reruns must not expose an earlier result.
result_record = Path(tempfile.gettempdir()) / f"sketch-result-{uuid.uuid4().hex}.json"
inference_code = """
import json, sys, logging
from pathlib import Path
from backend.sketch_refiner import refine_sketch
logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')
path = refine_sketch(sys.argv[1], sys.argv[2], sys.argv[3])
assert Path(path).is_file()
Path(sys.argv[4]).write_text(json.dumps({'path': path}))
"""
run_checked([sys.executable, "-u", "-c", inference_code, str(SKETCH), PROMPT, str(OUTPUT_DIR), str(result_record)],
            cwd=str(PROJECT), timeout=1800)
REFINED = Path(json.loads(result_record.read_text())["path"])
if not REFINED.is_file():
    raise FileNotFoundError("Refined PNG is missing")
print("Generated PNG:", REFINED)
display(DisplayImage(filename=str(REFINED)))
generation_record = json.loads((REFINED.parent / "generation.json").read_text())
print(json.dumps(generation_record, indent=2))
print("Compare seat/backrest shape, leg arrangement and viewpoint with the original; do not proceed to 3D yet.")

In [ ]:
import ipywidgets as widgets
from google.colab import files
button = widgets.Button(description="Download refined PNG", layout=widgets.Layout(width="230px"))
status = widgets.Output()
def download_clicked(_):
    with status:
        if REFINED is None or not REFINED.is_file():
            raise FileNotFoundError("Run refinement successfully first")
        run_checked([sys.executable, "-c", "import sys; from PIL import Image; image=Image.open(sys.argv[1]); assert image.format == 'PNG'; image.verify()", str(REFINED)], timeout=60)
        files.download(str(REFINED))
button.on_click(download_clicked)
display(button, status)

## What remains to verify on Colab
This notebook is statically checked and refinement is mocked in local tests, not GPU-validated. Verify imports with the protected package versions, actual model access, FP16 CPU offload on T4, and one genuine generated PNG. CPU offload trades VRAM for RAM and transfer time; model downloads and RAM exhaustion can fail. No automatic dependency repair or broad upgrades are performed. Neither ControlNet nor a high control strength guarantees correct chair topology; reject added/missing legs, wrong silhouette or changed viewpoint. Preserve the input/cleaned/control images and generation.json for comparison. No trained weights, TripoSR integration or frontend changes are included in this milestone.